# Análise Individual: Cliente (Avaliação)

Este notebook apresenta uma análise exploratória com foco na avaliação dos clientes da base Olist.  
A análise utiliza o dataset tratado `olist_super_dataset.csv`, gerado a partir do pipeline de ETL do projeto.

**Arquivo fonte:** `../data/processed/olist_super_dataset.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="pastel")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 50)
pd.options.display.float_format = "{:,.2f}".format

## 1. Carregamento e preparação dos dados

In [ ]:
df = pd.read_csv("../data/processed/olist_super_dataset.csv")

colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date"
]

for col in colunas_data:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df_avaliacao = df[df["review_score"].notna()].copy()

print(f"Shape do dataset completo: {df.shape}")
print(f"Shape do recorte com review_score válido: {df_avaliacao.shape}")
print(f"Quantidade de pedidos avaliados: {df_avaliacao['order_id'].nunique()}")

df_avaliacao.head()

Análise inicial: o recorte considera apenas registros com `review_score` preenchido, garantindo consistência para a análise da experiência do cliente. A base já contém variáveis operacionais e financeiras relevantes para investigar o que influencia a satisfação.

## 2. Dados trabalhados e EDA inicial

In [ ]:
colunas_principais = [
    "order_id", "review_score", "tempo_entrega_dias", "atraso_entrega",
    "price", "freight_value", "payment_value", "customer_state",
    "product_category_name_english", "receita_liquida"
]

display(df_avaliacao[colunas_principais].head())
display(df_avaliacao[colunas_principais].describe(include="all").T)
display(df_avaliacao.isnull().sum().sort_values(ascending=False).head(15))

Análise: as variáveis mais importantes para essa task são a nota de avaliação, o tempo de entrega, a ocorrência de atraso, os valores financeiros do pedido e o contexto geográfico e categórico do cliente. Esse conjunto é suficiente para avaliar a satisfação sob múltiplas perspectivas.

## 3. Distribuição das notas de avaliação

In [ ]:
distribuicao_review = (
    df_avaliacao["review_score"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x=distribuicao_review.index.astype(str),
    y=distribuicao_review.values
)

plt.title("Distribuição das Notas de Avaliação")
plt.xlabel("Review Score")
plt.ylabel("Quantidade de Registros")

for i, v in enumerate(distribuicao_review.values):
    ax.text(i, v, f"{int(v)}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

Insight: a distribuição das notas mostra o padrão geral de satisfação da base. Esse gráfico ajuda a identificar se a experiência dos clientes é predominantemente positiva ou se existe concentração relevante de notas baixas.

## 4. KPIs gerais de satisfação do cliente

In [ ]:
qtd_pedidos_avaliados = df_avaliacao["order_id"].nunique()
avaliacao_media = df_avaliacao["review_score"].mean()
avaliacao_mediana = df_avaliacao["review_score"].median()
percentual_nota_5 = (df_avaliacao["review_score"].eq(5).mean()) * 100
percentual_nota_1_2 = (df_avaliacao["review_score"].isin([1, 2]).mean()) * 100
tempo_medio_entrega = df_avaliacao["tempo_entrega_dias"].mean()
percentual_atraso = df_avaliacao["atraso_entrega"].mean() * 100

print("Resumo de KPIs - Avaliação do Cliente")
print(f"Quantidade de pedidos avaliados: {qtd_pedidos_avaliados}")
print(f"Avaliação média: {avaliacao_media:.2f}")
print(f"Avaliação mediana: {avaliacao_mediana:.2f}")
print(f"Percentual de nota 5: {percentual_nota_5:.2f}%")
print(f"Percentual de notas 1 ou 2: {percentual_nota_1_2:.2f}%")
print(f"Tempo médio de entrega: {tempo_medio_entrega:.2f} dias")
print(f"Percentual de pedidos com atraso: {percentual_atraso:.2f}%")

Análise: esses indicadores sintetizam o comportamento médio da satisfação na base e funcionam como referência para os recortes comparativos que aparecem nas próximas seções.

## 5. Impacto do atraso na avaliação

In [ ]:
avaliacao_por_atraso = (
    df_avaliacao.groupby("atraso_entrega")["review_score"]
    .mean()
    .rename(index={False: "No Prazo", True: "Atrasado"})
    .reset_index()
)

plt.figure(figsize=(8, 6))
ax = sns.barplot(
    data=avaliacao_por_atraso,
    x="atraso_entrega",
    y="review_score"
)

plt.title("Avaliação Média por Status de Entrega")
plt.xlabel("Status de Entrega")
plt.ylabel("Review Score Médio")
plt.ylim(0, 5)

for i, v in enumerate(avaliacao_por_atraso["review_score"].values):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

Insight técnico: essa comparação mostra se o atraso está associado a pior percepção do cliente. É um dos recortes mais importantes da análise porque conecta experiência do consumidor e eficiência logística.

## 6. Tempo de entrega e avaliação média

In [ ]:
faixas_entrega = pd.cut(
    df_avaliacao["tempo_entrega_dias"],
    bins=[-1, 3, 7, 14, 21, 60],
    labels=["Até 3 dias", "4-7 dias", "8-14 dias", "15-21 dias", "22+ dias"]
)

avaliacao_por_entrega = (
    df_avaliacao.assign(faixa_entrega=faixas_entrega)
    .groupby("faixa_entrega", observed=False)["review_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=avaliacao_por_entrega,
    x="faixa_entrega",
    y="review_score"
)

plt.title("Avaliação Média por Faixa de Tempo de Entrega")
plt.xlabel("Faixa de Tempo de Entrega")
plt.ylabel("Review Score Médio")
plt.ylim(0, 5)

for i, v in enumerate(avaliacao_por_entrega["review_score"].values):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

Insight: o tempo de entrega ajuda a entender se a satisfação piora de forma gradual à medida que o cliente espera mais tempo para receber o pedido.

## 7. Avaliação média por estado do cliente

In [ ]:
top_estados = (
    df_avaliacao.groupby("customer_state")["order_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .index
)

avaliacao_estado = (
    df_avaliacao[df_avaliacao["customer_state"].isin(top_estados)]
    .groupby("customer_state")["review_score"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x=avaliacao_estado.values,
    y=avaliacao_estado.index
)

plt.title("Avaliação Média nos 10 Estados com Mais Pedidos Avaliados")
plt.xlabel("Review Score Médio")
plt.ylabel("Estado do Cliente")
plt.xlim(0, 5)

for i, v in enumerate(avaliacao_estado.values):
    ax.text(v, i, f"  {v:.2f}", va="center")

plt.tight_layout()
plt.show()

Insight: esse recorte mostra se existem diferenças relevantes de experiência do cliente entre os principais estados da base, o que pode sinalizar desigualdades logísticas ou operacionais.

## 8. Avaliação média por categoria de produto

In [ ]:
top_categorias = (
    df_avaliacao.groupby("product_category_name_english")["order_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .index
)

avaliacao_categoria = (
    df_avaliacao[df_avaliacao["product_category_name_english"].isin(top_categorias)]
    .groupby("product_category_name_english")["review_score"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 6))
ax = sns.barplot(
    x=avaliacao_categoria.values,
    y=avaliacao_categoria.index
)

plt.title("Avaliação Média nas 10 Categorias com Mais Pedidos Avaliados")
plt.xlabel("Review Score Médio")
plt.ylabel("Categoria")
plt.xlim(0, 5)

for i, v in enumerate(avaliacao_categoria.values):
    ax.text(v, i, f"  {v:.2f}", va="center")

plt.tight_layout()
plt.show()

Insight: a experiência do cliente pode variar conforme o tipo de produto comprado. Esse gráfico ajuda a identificar categorias com avaliação mais crítica e categorias mais estáveis.

## 9. Avaliação por faixa de valor do pedido

In [ ]:
base_valor = (
    df_avaliacao.groupby("order_id")
    .agg(
        payment_value=("payment_value", "mean"),
        review_score=("review_score", "mean")
    )
    .dropna()
)

base_valor["faixa_valor"] = pd.qcut(
    base_valor["payment_value"],
    q=4,
    labels=["Baixo", "Médio-Baixo", "Médio-Alto", "Alto"],
    duplicates="drop"
)

avaliacao_valor = (
    base_valor.groupby("faixa_valor", observed=False)["review_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=avaliacao_valor,
    x="faixa_valor",
    y="review_score"
)

plt.title("Avaliação Média por Faixa de Valor do Pedido")
plt.xlabel("Faixa de Valor")
plt.ylabel("Review Score Médio")
plt.ylim(0, 5)

for i, v in enumerate(avaliacao_valor["review_score"].values):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

Insight: essa análise investiga se pedidos de maior valor apresentam experiência percebida diferente, o que é importante para compreender a sensibilidade do cliente ao ticket pago.

## 10. Correlações com outros parâmetros relacionados

In [ ]:
base_correlacao = (
    df_avaliacao.groupby("order_id")
    .agg(
        review_medio=("review_score", "mean"),
        tempo_entrega=("tempo_entrega_dias", "mean"),
        atraso=("atraso_entrega", "mean"),
        frete_medio=("freight_value", "mean"),
        valor_pago=("payment_value", "mean"),
        valor_produto=("price", "mean")
    )
    .dropna()
)

corr_plot = base_correlacao.corr()

corr_plot

In [ ]:
corr_review = corr_plot["review_medio"].drop("review_medio").sort_values()

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x=corr_review.values,
    y=corr_review.index
)

plt.title("Correlação das Variáveis com a Avaliação Média")
plt.xlabel("Coeficiente de Correlação")
plt.ylabel("")

for i, v in enumerate(corr_review.values):
    if v >= 0:
        ax.text(v, i, f"  {v:.2f}", va="center")
    else:
        ax.text(v, i, f"{v:.2f}  ", va="center", ha="right")

plt.tight_layout()
plt.show()

In [ ]:
base_correlacao["atraso_pct"] = base_correlacao["atraso"] * 100

plt.figure(figsize=(10, 6))
sns.regplot(
    data=base_correlacao,
    x="atraso_pct",
    y="review_medio",
    scatter_kws={"alpha": 0.25, "s": 25},
    line_kws={"linewidth": 2}
)

plt.title("Relação entre Taxa de Atraso e Avaliação Média por Pedido")
plt.xlabel("Taxa de Atraso (%)")
plt.ylabel("Review Score Médio")
plt.xlim(left=0)
plt.ylim(1, 5)
plt.tight_layout()
plt.show()

Insight técnico: as correlações ajudam a quantificar a relação entre logística, valor financeiro e satisfação. Em especial, o atraso tende a apresentar associação negativa com a nota de avaliação.

## 11. Agrupamento por faixa de avaliação

In [ ]:
base_correlacao["faixa_avaliacao"] = pd.cut(
    base_correlacao["review_medio"],
    bins=[0, 2, 3, 4, 5],
    labels=["Baixa (0-2]", "Média-Baixa (2-3]", "Média-Alta (3-4]", "Alta (4-5]"],
    include_lowest=True
)

resumo_faixas = (
    base_correlacao.groupby("faixa_avaliacao", observed=False)
    .agg(
        pedidos=("review_medio", "count"),
        tempo_entrega_medio=("tempo_entrega", "mean"),
        taxa_atraso_media=("atraso", "mean"),
        frete_medio=("frete_medio", "mean"),
        valor_pago_medio=("valor_pago", "mean")
    )
)

resumo_faixas["taxa_atraso_media"] = resumo_faixas["taxa_atraso_media"] * 100
resumo_faixas

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=resumo_faixas.reset_index(),
    x="faixa_avaliacao",
    y="tempo_entrega_medio"
)

plt.title("Tempo Médio de Entrega por Faixa de Avaliação")
plt.xlabel("Faixa de Avaliação")
plt.ylabel("Tempo Médio de Entrega (dias)")

for i, v in enumerate(resumo_faixas["tempo_entrega_medio"].values):
    ax.text(i, v, f"{v:.1f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=resumo_faixas.reset_index(),
    x="faixa_avaliacao",
    y="taxa_atraso_media"
)

plt.title("Taxa Média de Atraso por Faixa de Avaliação")
plt.xlabel("Faixa de Avaliação")
plt.ylabel("Taxa Média de Atraso (%)")

for i, v in enumerate(resumo_faixas["taxa_atraso_media"].values):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom")

plt.tight_layout()
plt.show()

Insight: o agrupamento por faixa de avaliação permite comparar padrões operacionais entre pedidos melhor e pior avaliados. Isso ajuda a transformar a análise em uma leitura mais gerencial e menos apenas descritiva.

## 12. Principais insights da análise

A análise da avaliação dos clientes mostra que a satisfação não depende apenas do produto comprado, mas também da experiência operacional associada ao pedido. O comportamento das notas indica predominância de avaliações positivas, mas a distribuição não é homogênea entre todos os contextos da base.

Os recortes de atraso e tempo de entrega sugerem que a logística exerce papel central na percepção do cliente. Pedidos entregues fora do prazo ou com maior tempo de espera tendem a apresentar pior avaliação média.

Além disso, os recortes por estado, categoria e faixa de valor mostram que a experiência do cliente pode variar conforme o contexto da compra. Isso reforça a necessidade de analisar satisfação de forma segmentada, e não apenas como uma média geral.

## 13. Validação da conclusão

In [ ]:
validacao = resumo_faixas[[
    "pedidos",
    "tempo_entrega_medio",
    "taxa_atraso_media",
    "frete_medio",
    "valor_pago_medio"
]].copy()

validacao

Validação: ao comparar as faixas de avaliação, observa-se que pedidos com notas mais baixas tendem a apresentar piores indicadores operacionais, especialmente em tempo de entrega e atraso. Isso sustenta a conclusão de que a experiência logística é um dos principais fatores associados à percepção do cliente.

## 14. Conclusão final

A análise individual de cliente com foco em avaliação mostrou que a satisfação na base Olist está fortemente relacionada à execução da jornada de entrega. Atraso e maior tempo de espera aparecem como fatores críticos para redução da nota média.

Como proposta de valor para o negócio, esse tipo de análise pode apoiar ações de melhoria operacional, priorização de rotas e monitoramento de categorias ou regiões com maior risco de insatisfação. Dessa forma, a empresa pode atuar de forma preventiva na experiência do consumidor.